# Reproducing ResNet on CIFAR-10 (Kaggle GPU)

Kaggle port of `notebooks/reproduce_colab.ipynb`. Same repo, same driver, same schedule.

**Settings > Accelerator > GPU T4 x2 (or P100) before running.**

Run this with **Save Version > Save & Run All (Commit)**: it executes headless, so no browser
tab and no awake laptop are needed, and `/kaggle/working/results.csv` is persisted with the
version. Seeds already present in the repo's committed `results.csv` are skipped.

In [ ]:
!git clone -q https://github.com/AmroAbujabal/resnet-cifar-repro.git /kaggle/working/repo
%cd /kaggle/working/repo
!pip -q install pyyaml pytest
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# Results live in /kaggle/working (persisted with the saved version), seeded from the repo's
# committed results.csv so the seeds already reported are carried forward and skipped.
import csv, os, shutil
RESULTS = '/kaggle/working/results.csv'
# Per-run eval logs are committed with the repo, so they must land outside the
# clone (which is thrown away) and be copied back into git afterwards.
LOGS = '/kaggle/working/logs'
if not os.path.exists(RESULTS):
    shutil.copy('results.csv', RESULTS)

def done(model, seed):
    with open(RESULTS) as f:
        return any(r['model'] == model and int(r['seed']) == seed for r in csv.DictReader(f))

print(open(RESULTS).read())

In [ ]:
# Full suite, including the T1 data tests that download CIFAR-10 (and CIFAR-100
# for the two Phase 3 tests) -- the Toronto mirror is slow, allow ~15 min.
!python -m pytest -q


In [ ]:
# ResNet-20, seeds 0-2 (paper Table 6: 8.75%). ~51 min/seed on a T4.
for s in (0, 1, 2):
    if done('resnet20', s):
        print(f'skip resnet20 seed {s} -- already in results.csv')
        continue
    !python scripts/train.py --config configs/resnet20.yaml --seed {s} --device cuda --results {RESULTS} --log-dir {LOGS}

In [ ]:
# ResNet-56, seeds 0-2 (paper Table 6: 6.97%). ~2.5 h/seed on a T4 -- check this fits the
# session limit before committing; if not, run seeds one at a time across versions.
for s in (0, 1, 2):
    if done('resnet56', s):
        print(f'skip resnet56 seed {s} -- already in results.csv')
        continue
    !python scripts/train.py --config configs/resnet56.yaml --seed {s} --device cuda --results {RESULTS} --log-dir {LOGS}

## Phase 3 — pre-activation × CIFAR-100 (2×2)

Four cells: {original, pre-act} ResNet-56 × {CIFAR-10, CIFAR-100}. **original × CIFAR-10 is
already done** (3 seeds in `results.csv`), so three remain, ~6.3 h each (3 seeds × ~2.1 h).

Kaggle kills a session at its limit and **discards `/kaggle/working`**, so run **one config per
saved version**: set `PHASE3` below, Save & Run All, then pull the output, commit `results.csv`,
push, and come back for the next one.


In [ ]:
# Set this per version -- do NOT run more than one config in a single version.
PHASE3 = 'preact56_c100'     # done: preact56, resnet56_c100.  this is the last cell

for s in (0, 1, 2):
    if done(PHASE3, s):
        print(f'skip {PHASE3} seed {s} -- already in results.csv')
        continue
    !python scripts/train.py --config configs/{PHASE3}.yaml --seed {s} --device cuda --results {RESULTS} --log-dir {LOGS}


## Phase 4 — rerun of original ResNet-56 on CIFAR-10, seed 2

Phase 2/3 kept no evaluation logs, so the seed 2 excursion (8.22%, 1.31 points above seed 0) has
no train-error curve behind it and cannot be read as an optimisation failure or a generalisation
one. This rerun is the instrument fix: deterministic seeding, train + test error logged every
8,000 iterations to `logs/resnet56_rerun_seed2.csv`.

It appends as `resnet56_rerun`, **not** as a fourth `resnet56` seed. The reproduction claim was
pre-registered on three seeds and keeps its mean and s.d.

**It will not reproduce 8.22% except by coincidence.** The original runs used autotuned cuDNN
kernels; determinism changes kernel selection and therefore the numerics from the first step. A
different number here is a finding about the original runs, not a failure of this one.


In [ ]:
# ~2.2-3 h on a T4 (deterministic cuDNN kernels can be slower than autotuned ones).
for s in (2,):
    if done('resnet56_rerun', s):
        print(f'skip resnet56_rerun seed {s} -- already in results.csv')
        continue
    !python scripts/train.py --config configs/resnet56_rerun.yaml --seed {s} --device cuda --results {RESULTS} --log-dir {LOGS}


In [ ]:
import pandas as pd
df = pd.read_csv(RESULTS)
display(df)
print(df.groupby('model')[['test_error_pct', 'train_error_pct']].agg(['mean', 'std', 'count']))
